In [2]:
# CK
# Load the local student records.
from pathlib import Path
import pandas as pd

paths = [
    Path("students_cleaned.csv"), Path("data.csv"),
    Path("../data/processed/students_cleaned.csv"), Path("../data/raw/data.csv"),
    Path("data/processed/students_cleaned.csv"), Path("data/raw/data.csv"),
]
for file in paths:
    if file.is_file():
        break
else:
    raise FileNotFoundError(f"Open this notebook beside data.csv or students_cleaned.csv. Current folder: {Path.cwd()}")

df_clean = pd.read_csv(file, sep=None, engine="python", encoding="utf-8-sig")
df_clean.columns = (df_clean.columns.str.strip().str.lower()
                    .str.replace(r"[ /]+", "_", regex=True)
                    .str.replace(r"['()]", "", regex=True))
print("Loaded:", file, "| Rows and columns:", df_clean.shape)

Loaded: data.csv | Rows and columns: (4424, 37)


In [4]:
# BV
# Count how often each number of credited units appears.
df_clean["curricular_units_2nd_sem_credited"].value_counts()

curricular_units_2nd_sem_credited
0     3894
1      107
2       92
4       78
5       68
3       49
6       26
11      20
7       16
9       15
12      14
10      13
8       12
13       9
14       4
15       2
16       2
18       2
19       1
Name: count, dtype: int64

In [5]:
# BV
# See the typical course load and the range of second-semester registrations.
df_clean["curricular_units_2nd_sem_enrolled"].describe()

count    4424.000000
mean        6.232143
std         2.195951
min         0.000000
25%         5.000000
50%         6.000000
75%         7.000000
max        23.000000
Name: curricular_units_2nd_sem_enrolled, dtype: float64

In [6]:
# BV
# See the typical number and range of evaluations.
df_clean["curricular_units_2nd_sem_evaluations"].describe()

count    4424.000000
mean        8.063291
std         3.947951
min         0.000000
25%         6.000000
50%         8.000000
75%        10.000000
max        33.000000
Name: curricular_units_2nd_sem_evaluations, dtype: float64

In [15]:
# BV
# Summarize how many units students passed in semester two.
df_clean["curricular_units_2nd_sem_approved"].describe()

count    4424.000000
mean        4.435805
std         3.014764
min         0.000000
25%         2.000000
50%         5.000000
75%         6.000000
max        20.000000
Name: curricular_units_2nd_sem_approved, dtype: float64

In [16]:
# BV
#Compare the average number of units passed across the three recorded outcomes.
df_clean.groupby("target")["curricular_units_2nd_sem_approved"].mean()

target
Dropout     1.940183
Enrolled    4.057935
Graduate    6.177003
Name: curricular_units_2nd_sem_approved, dtype: float64

In [17]:
# BV
# Check the distribution and range of average semester-two grades.
df_clean["curricular_units_2nd_sem_grade"].describe()

count    4424.000000
mean       10.230206
std         5.210808
min         0.000000
25%        10.750000
50%        12.200000
75%        13.333333
max        18.571429
Name: curricular_units_2nd_sem_grade, dtype: float64

In [18]:
# BV
# Count the students with a recorded grade of zero.
(df_clean["curricular_units_2nd_sem_grade"] == 0).sum()

np.int64(870)

In [19]:
# BV
# See how often a student has units without evaluations.
df_clean["curricular_units_2nd_sem_without_evaluations"].value_counts()

curricular_units_2nd_sem_without_evaluations
0     4142
1      140
2       48
3       35
4       21
5       17
6        8
8        6
7        5
12       2
Name: count, dtype: int64

In [20]:
# BV
# Check whether students who passed more units in semester one also tended to pass more in semester two.
df_clean[["curricular_units_1st_sem_approved", "curricular_units_2nd_sem_approved"]].corr()

,curricular_units_1st_sem_approved,curricular_units_2nd_sem_approved
curricular_units_1st_sem_approved,1.000000,0.904002
curricular_units_2nd_sem_approved,0.904002,1.000000


In [21]:
# BV
# Count how many different unemployment, inflation and GDP values appear.
df_clean[["unemployment_rate", "inflation_rate", "gdp"]].nunique()

unemployment_rate    10
inflation_rate        9
gdp                  10
dtype: int64

In [23]:
# BV
# Count how many students share each unemployment rate.
df_clean["unemployment_rate"].value_counts()
#NOTE: Found 870 zero grades. Average semester-two passes were 1.94 for Dropout, 4.06 for Enrolled and 6.18 for Graduate. The correlation between semester pass counts was 0.904. 
#NOTE: These were descriptive checks across the whole file. CK’s follow-up below focuses on interpreting unusual values and finishing the counts.

unemployment_rate
7.6     571
9.4     533
10.8    525
12.4    445
12.7    419
11.1    414
15.5    397
13.9    390
8.9     368
16.2    362
Name: count, dtype: int64

In [24]:
# CK
#Keep a working copy of just my nine assigned fields and retain the original row numbers.
prefix = "curricular_units_2nd_sem_"
semester = [prefix + name for name in [
    "credited", "enrolled", "evaluations", "approved", "grade", "without_evaluations"
]]
economy = ["unemployment_rate", "inflation_rate", "gdp"]
my_columns = semester + economy
cem = df_clean[my_columns].copy()
print("Student rows, assigned columns:", cem.shape)

Student rows, assigned columns: (4424, 9)


In [25]:
# CK
# Check all nine fields for blanks, unexpected text and zeroes before deciding whether any cleaning is justified.
numeric = cem.apply(pd.to_numeric, errors="coerce")
quality = pd.DataFrame({
    "missing_cells": cem.isna().sum(),
    "unexpected_text": (numeric.isna() & cem.notna()).sum(),
    "distinct_values": cem.nunique(),
    "zeroes": numeric.eq(0).sum(),
})
print(quality.to_string())

                                              missing_cells  unexpected_text  distinct_values  zeroes
curricular_units_2nd_sem_credited                         0                0               19    3894
curricular_units_2nd_sem_enrolled                         0                0               22     180
curricular_units_2nd_sem_evaluations                      0                0               30     401
curricular_units_2nd_sem_approved                         0                0               20     870
curricular_units_2nd_sem_grade                            0                0              786     870
curricular_units_2nd_sem_without_evaluations              0                0               10    4142
unemployment_rate                                         0                0               10       0
inflation_rate                                            0                0                9       0
gdp                                                       0                0      

In [28]:
# CK
# WHY: Count the six distributions Bindu had not yet tabulated, including inflation and GDP.
remaining_counts = [
    prefix + "enrolled", prefix + "evaluations", prefix + "approved",
    prefix + "grade", "inflation_rate", "gdp",
]
frequencies = {}
for column in remaining_counts:
    counts = cem[column].value_counts(dropna=False)
    frequencies[column] = counts
    print("\n" + column)
    if column == prefix + "grade":
        print(counts.head(10).to_string())
    else:
        print(counts.to_string())


curricular_units_2nd_sem_enrolled
curricular_units_2nd_sem_enrolled
6     1913
5     1054
8      661
7      304
0      180
11      60
9       50
10      48
12      44
13      37
14      22
4       17
17      12
2        5
19       3
3        3
1        3
15       2
23       2
18       2
16       1
21       1

curricular_units_2nd_sem_evaluations
curricular_units_2nd_sem_evaluations
8     792
6     614
7     563
9     456
0     401
10    355
5     288
11    255
12    226
13    126
14     98
15     73
16     49
17     25
18     22
19     19
4      10
21     10
22     10
20      8
2       4
23      4
1       3
26      3
24      3
3       2
27      2
28      1
25      1
33      1

curricular_units_2nd_sem_approved
curricular_units_2nd_sem_approved
6     965
0     870
5     726
4     414
7     331
8     321
3     285
2     198
1     114
11     48
10     38
9      36
12     34
13     21
17      8
14      6
19      3
20      2
16      2
18      2

curricular_units_2nd_sem_grade
curricular_un

In [31]:
# To see every grade frequency
print(frequencies[prefix + "grade"].to_string())

curricular_units_2nd_sem_grade
0.000000     870
12.000000    170
11.000000    165
13.000000    150
11.500000     86
12.500000     84
10.000000     77
14.000000     77
13.500000     65
12.666667     61
11.333333     59
12.333333     56
11.666667     54
13.666667     47
13.333333     47
12.200000     45
11.800000     45
12.600000     45
10.500000     45
11.600000     44
11.250000     44
12.833333     42
12.800000     42
12.400000     41
12.166667     36
13.200000     35
11.400000     34
12.250000     33
14.166667     32
11.750000     30
10.666667     29
11.833333     28
15.000000     27
12.750000     25
14.333333     24
11.200000     24
13.166667     24
14.500000     24
13.400000     23
13.800000     21
13.250000     21
14.200000     20
13.285714     20
10.750000     20
10.333333     19
13.833333     19
14.666667     19
13.600000     18
16.000000     17
12.142857     17
14.600000     16
13.750000     15
12.857143     14
12.285714     14
11.857143     14
13.142857     14
12.428571     13


In [32]:
# CK
# Complete describe() for the five fields Bindu had not summarized and put them side by side.
remaining_summaries = [prefix + "credited", prefix + "without_evaluations"] + economy
print(numeric[remaining_summaries].describe().T.round(2).to_string())

                                               count   mean   std   min  25%    50%    75%    max
curricular_units_2nd_sem_credited             4424.0   0.54  1.92  0.00  0.0   0.00   0.00  19.00
curricular_units_2nd_sem_without_evaluations  4424.0   0.15  0.75  0.00  0.0   0.00   0.00  12.00
unemployment_rate                             4424.0  11.57  2.66  7.60  9.4  11.10  13.90  16.20
inflation_rate                                4424.0   1.23  1.38 -0.80  0.3   1.40   2.60   3.70
gdp                                           4424.0   0.00  2.27 -4.06 -1.7   0.32   1.79   3.51


In [33]:
# CK
# Check for impossible counts or grades, and verify whether grade averages contain decimals.
grade = numeric[prefix + "grade"]
count_fields = [column for column in semester if column != prefix + "grade"]
print("Negative counts:", int(numeric[count_fields].lt(0).sum().sum()))
print("Fractional counts:", int((numeric[count_fields].mod(1).ne(0) & numeric[count_fields].notna()).sum().sum()))
print("Grades outside 0–20:", ((grade < 0) | (grade > 20)).sum())
print("Grades with decimals:", grade.dropna().mod(1).ne(0).sum())

Negative counts: 0
Fractional counts: 0
Grades outside 0–20: 0
Grades with decimals: 2869


In [34]:
# CK
# Follow up on 870 zero grades by checking who registered, passed or had an evaluation.
registered = numeric[prefix + "enrolled"]
passed = numeric[prefix + "approved"]
evaluations = numeric[prefix + "evaluations"]
no_eval = numeric[prefix + "without_evaluations"]

print("Zero grade:", grade.eq(0).sum())
print("Zero grade AND zero passed:", (grade.eq(0) & passed.eq(0)).sum())
print("Zero grade AND zero registered:", (grade.eq(0) & registered.eq(0)).sum())
print("Zero grade AND registered for one or more:", (grade.eq(0) & registered.gt(0)).sum())
print("Zero grade AND at least one evaluation:", (grade.eq(0) & evaluations.gt(0)).sum())
print("Passed more units than registered:", passed.gt(registered).sum())

Zero grade: 870
Zero grade AND zero passed: 870
Zero grade AND zero registered: 180
Zero grade AND registered for one or more: 690
Zero grade AND at least one evaluation: 469
Passed more units than registered: 0


In [35]:
# CK
# WHY: Show Bindu the specific records where units without evaluations exceed the units registered.
flagged = no_eval > registered
review = numeric.loc[flagged, [prefix + "enrolled", prefix + "evaluations", prefix + "without_evaluations"]]
print(review.to_string())

      curricular_units_2nd_sem_enrolled  curricular_units_2nd_sem_evaluations  curricular_units_2nd_sem_without_evaluations
533                                   6                                    12                                            12
1378                                  6                                     7                                             7
2931                                  5                                    14                                             8


In [36]:
# CK
# Check whether negative or repeated economic values need any correction.
economic_check = pd.DataFrame({
    "distinct": cem[economy].nunique(),
    "minimum": numeric[economy].min(),
    "maximum": numeric[economy].max(),
    "negative_rows": numeric[economy].lt(0).sum(),
})
print(economic_check.to_string())

                   distinct  minimum  maximum  negative_rows
unemployment_rate        10     7.60    16.20              0
inflation_rate            9    -0.80     3.70            923
gdp                      10    -4.06     3.51           1711


In [37]:
# CK
# WHY: Give Bindu the quality checks, observed ranges and a decision for each of the nine fields.
action = {
    semester[0]: "Keep zero as zero credited units.",
    semester[1]: "Keep zero as zero registrations.",
    semester[2]: "Keep evaluation counts; multiple evaluations per unit are possible.",
    semester[3]: "Keep zero passes; no pass count exceeds registrations.",
    semester[4]: "Keep decimal grades and recorded zeroes.",
    semester[5]: "Keep values; check the 3 rows where this count exceeds enrolled units.",
    "unemployment_rate": "Keep the 10 observed rates.",
    "inflation_rate": "Keep the 9 observed rates, including negative inflation.",
    "gdp": "Keep the 10 observed values in source units, including negatives.",
}
summary = numeric.describe().T
handoff = quality.join(summary[["count", "min", "50%", "max"]]).rename(columns={"50%": "median"})
handoff["action"] = pd.Series(action)
handoff.index.name = "column"
print(handoff[["median", "min", "max", "action"]].round(2).to_string())

                                              median   min    max                                                                  action
column                                                                                                                                   
curricular_units_2nd_sem_credited               0.00  0.00  19.00                                       Keep zero as zero credited units.
curricular_units_2nd_sem_enrolled               6.00  0.00  23.00                                        Keep zero as zero registrations.
curricular_units_2nd_sem_evaluations            8.00  0.00  33.00     Keep evaluation counts; multiple evaluations per unit are possible.
curricular_units_2nd_sem_approved               5.00  0.00  20.00                  Keep zero passes; no pass count exceeds registrations.
curricular_units_2nd_sem_grade                 12.20  0.00  18.57                                Keep decimal grades and recorded zeroes.
curricular_units_2nd_sem_without_e

In [39]:
#CK
# Save the checks for assigned columns for audit trail
handoff.to_csv("Cem_Column_Checks.csv")